# Control T13 · Conector de Ingesta API

Este cuaderno levanta la API mock en background, ejecuta el conector
contra distintos perfiles de equipo, y genera los archivos de evidencia.

In [ ]:
# Celda 1 — Dependencias e importaciones
import subprocess
import sys
import time
import json
import os
from pathlib import Path
from datetime import datetime

# Instalar dependencias si no están presentes
subprocess.run(
    [sys.executable, "-m", "pip", "install", "--quiet",
     "psycopg2-binary", "flask", "requests"],
    check=True
)

print("Dependencias instaladas.")
print(f"Directorio de trabajo: {os.getcwd()}")

In [ ]:
# Celda 2 — Equipo 1 (control): 200 siempre, 3 páginas

MOCK_PORT_1 = 15001
MOCK_SCRIPT = "practica/s13-conector-api/api_mock.py"
CONNECTOR   = "-m src.ingesta.conector_api"
API_URL_TPL = "http://127.0.0.1:{port}/api/medidores"
TOKEN       = "demo-token"

# Levantar mock equipo-1
proc_mock_1 = subprocess.Popen(
    [sys.executable, MOCK_SCRIPT, "--equipo", "equipo-1", "--port", str(MOCK_PORT_1)],
    stdout=subprocess.PIPE, stderr=subprocess.PIPE
)
time.sleep(2)  # dar tiempo a Flask para arrancar

result_1 = subprocess.run(
    [sys.executable] + CONNECTOR.split() +
    ["--url", API_URL_TPL.format(port=MOCK_PORT_1),
     "--token", TOKEN],
    capture_output=True, text=True
)

proc_mock_1.terminate()
proc_mock_1.wait()

print("=== EQUIPO-1 (control) ===")
print("stdout:", result_1.stdout)
if result_1.stderr:
    print("stderr:", result_1.stderr[:500])

try:
    output_equipo1 = json.loads(result_1.stdout)
except json.JSONDecodeError:
    output_equipo1 = {"raw": result_1.stdout}

print("Resultado parseado:", output_equipo1)

In [ ]:
# Celda 3 — Equipo 2: páginas 1-2 OK, página 3 → 429 x2 luego 200

MOCK_PORT_2 = 15002

proc_mock_2 = subprocess.Popen(
    [sys.executable, MOCK_SCRIPT, "--equipo", "equipo-2", "--port", str(MOCK_PORT_2)],
    stdout=subprocess.PIPE, stderr=subprocess.PIPE
)
time.sleep(2)

result_2 = subprocess.run(
    [sys.executable] + CONNECTOR.split() +
    ["--url", API_URL_TPL.format(port=MOCK_PORT_2),
     "--token", TOKEN,
     "--max-intentos", "3",
     "--t0", "0.1"],
    capture_output=True, text=True
)

proc_mock_2.terminate()
proc_mock_2.wait()

print("=== EQUIPO-2 (429 → reintento) ===")
print("stdout:", result_2.stdout)
if result_2.stderr:
    print("stderr:", result_2.stderr[:500])

try:
    output_equipo2 = json.loads(result_2.stdout)
except json.JSONDecodeError:
    output_equipo2 = {"raw": result_2.stdout}

print("Resultado parseado:", output_equipo2)

In [ ]:
# Celda 4 — Equipo 3: página 2 → 500 siempre (agotado)

MOCK_PORT_3 = 15003

proc_mock_3 = subprocess.Popen(
    [sys.executable, MOCK_SCRIPT, "--equipo", "equipo-3", "--port", str(MOCK_PORT_3)],
    stdout=subprocess.PIPE, stderr=subprocess.PIPE
)
time.sleep(2)

result_3 = subprocess.run(
    [sys.executable] + CONNECTOR.split() +
    ["--url", API_URL_TPL.format(port=MOCK_PORT_3),
     "--token", TOKEN,
     "--max-intentos", "3",
     "--t0", "0.1"],
    capture_output=True, text=True
)

proc_mock_3.terminate()
proc_mock_3.wait()

print("=== EQUIPO-3 (500 agotado) ===")
print("stdout:", result_3.stdout)
if result_3.stderr:
    print("stderr:", result_3.stderr[:500])

try:
    output_equipo3 = json.loads(result_3.stdout)
except json.JSONDecodeError:
    output_equipo3 = {"raw": result_3.stdout}

print("Resultado parseado:", output_equipo3)

In [ ]:
# Celda 5 — Generar archivos de evidencia

EVIDENCIA_DIR = Path("practica/s13-conector-api")
EVIDENCIA_DIR.mkdir(parents=True, exist_ok=True)

fecha_hoy = datetime.utcnow().strftime("%Y-%m-%d %H:%M UTC")
TOTAL_REGISTROS_INFORMATIVO = 15  # según el mock

def validar_conciliacion(output: dict, label: str) -> str:
    estado = output.get('estado_corrida', 'DESCONOCIDO')
    insertados = output.get('registros_insertados', 0)
    actualizados = output.get('registros_actualizados', 0)
    total_en_stg = insertados + actualizados
    paginas = output.get('paginas_procesadas', 0)
    concilia = total_en_stg == TOTAL_REGISTROS_INFORMATIVO
    return f"""
### {label}
| Campo | Valor |
|---|---|
| `estado_corrida` | `{estado}` |
| `paginas_procesadas` | {paginas} |
| `registros_insertados` | {insertados} |
| `registros_actualizados` | {actualizados} |
| `total_en_stg` | {total_en_stg} |
| `total_registros_informativo` | {TOTAL_REGISTROS_INFORMATIVO} |
| Conciliación | {'✅ OK' if concilia else '⚠️ DIFERENCIA'} |
| `estado_corrida = COMPLETA` | {'✅ Validado' if estado == 'COMPLETA' else '❌ No es COMPLETA'} |
"""

# evidencia_nivel2_equipo-1.md
contenido_e1 = f"""# Evidencia Nivel 2 — Equipo 1 (control)

Generado: {fecha_hoy}  
Equipo: equipo-1  
Perfil: control (200 siempre, 3 páginas de 5 registros)

{validar_conciliacion(output_equipo1, 'Corrida equipo-1')}

## Salida completa del conector

```json
{json.dumps(output_equipo1, indent=2, ensure_ascii=False)}
```
"""
Path(EVIDENCIA_DIR / "evidencia_nivel2_equipo-1.md").write_text(contenido_e1, encoding="utf-8")
print("Generado: evidencia_nivel2_equipo-1.md")

# evidencia_nivel2_equipo-2.md
contenido_e2 = f"""# Evidencia Nivel 2 — Equipo 2

Generado: {fecha_hoy}  
Equipo: equipo-2  
Perfil: páginas 1-2 → 200, página 3 → 429 x2 luego 200

{validar_conciliacion(output_equipo2, 'Corrida equipo-2')}

## Salida completa del conector

```json
{json.dumps(output_equipo2, indent=2, ensure_ascii=False)}
```
"""
Path(EVIDENCIA_DIR / "evidencia_nivel2_equipo-2.md").write_text(contenido_e2, encoding="utf-8")
print("Generado: evidencia_nivel2_equipo-2.md")

print("\nResumen:")
print(f"  equipo-1 estado_corrida: {output_equipo1.get('estado_corrida')}")
print(f"  equipo-2 estado_corrida: {output_equipo2.get('estado_corrida')}")

In [ ]:
# Celda 6 — Los procesos mock ya fueron terminados en sus celdas respectivas.
# Esta celda confirma la limpieza y muestra un resumen final.

print("Todos los procesos mock han sido detenidos.")
print("\n=== Resumen de corridas ===")
for label, out in [("equipo-1", output_equipo1), ("equipo-2", output_equipo2), ("equipo-3", output_equipo3)]:
    print(f"  {label}: estado={out.get('estado_corrida')}, "
          f"páginas={out.get('paginas_procesadas')}, "
          f"insertados={out.get('registros_insertados')}, "
          f"actualizados={out.get('registros_actualizados')}")